# Outcome Variable Construction — Competing Risks Survival Analysis

Constructs two variables for each of the 201 UCDP conflicts:

- **`ever_agreement`** (0 / 1) — whether a PA-X agreement occurred within the observable GED spell
- **`termination_outcome_label`** — conflict's terminal state at the end of the observable spell

In [180]:
import sys, os
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 60)

PANEL_PATH = '../../data/output/conflict_level/conflict_panel.csv'
TERM_PATH  = '../../data/input/ucdp/UCDPConflictTerminationDataset_v4_2024_Conflict.csv'
OUT_PATH   = '../../data/output/conflict_level/conflict_cause.csv'

---
## **1. Define Scope of Analysis**

### Unit and estimand

The unit of analysis is **`conflict_id`** — a UCDP conflict that span multiple episodes of violence separated by dormancy periods. Dormancy (months with `best = 0`) is treated as a gap within the spell, not a competing event.

Each conflict enters the risk set at `start_date` (first GED event) and exits at the first PA-X agreement within the GED window, or end date of conflict id with termination outcome military victory or actor disappearance, or end of the observation window (censoring).

### Conflict panel structure (GED-based)

The panel is built from UCDP GED events aggregated to **conflict_id × month**, running from 1989-01 to 2024-12 (dormancy months retained as `best = 0`). GED applies an important inclusion rule (codebook, Section 3):

> "If a dyad crossed the 25-deaths threshold in a single year, but did generate some events in either previous or subsequent years, all events belonging to the dyad are included, including those in years where the threshold was not crossed."

This means `end_date` (last GED event month) frequently extends **years beyond** the last UCDP/PRIO active episode. The three datasets we use each define activity differently:

| Dataset | Activity criterion | What the end date captures |
|---|---|---|
| **UCDP/PRIO Conflict** | ≥ 25 battle deaths / calendar year | Last year an episode is "active" |
| **UCDP Conflict Termination** | Inherits UCDP/PRIO structure | `c_ep_endyear` = last year with ≥ 25 deaths |
| **UCDP GED** | All events for recognized dyads, including sub-threshold years | Last month with *any* GED event |

### PA-X merge and agreement timing

PA-X agreements are merged by `conflict_id`, giving `first_agreement_date` (month index of the first PA-X agreement; 0 = never) and `ever_agreement` (1 if ever signed). Agreements fall into three positions relative to the GED spell `[start_date, end_date]`:

| Timing | Definition | Treatment |
|---|---|---|
| **during_ged** | `start_date ≤ agreement ≤ end_date` | Signing event → `ever_agreement = 1` |
| **pre_ged** | `agreement < start_date` | Outside spell → reclassify `ever_agreement = 0` |
| **post_ged** | `agreement > end_date` | Outside spell → depends on termination outcome (Step 3) |

In [181]:
# ── Load conflict panel ───────────────────────────────────────────────────────

df_panel = pd.read_csv(PANEL_PATH, low_memory=False)

# Conflict-level summary (one row per conflict_id)
cl = (
    df_panel.drop_duplicates('conflict_id')
    [['conflict_id', 'country', 'isocode',
      'start_date', 'end_date', 'pax_first_date',
      'start_date_numeric', 'end_date_numeric',
      'ever_agreement', 'first_agreement_date']]
    .copy()
)

def agree_timing(row):
    if row['ever_agreement'] == 0:
        return 'never_signed'
    if row['first_agreement_date'] < row['start_date_numeric']:
        return 'pre_ged'
    if row['first_agreement_date'] > row['end_date_numeric']:
        return 'post_ged'
    return 'during_ged'

cl['agree_timing'] = cl.apply(agree_timing, axis=1)

print(f"Conflicts: {len(cl)}    |    "
      f"Panel: {df_panel['year_mo'].min()} – {df_panel['year_mo'].max()}")
print()
print("Agreement timing breakdown:")
for t, n in cl['agree_timing'].value_counts().items():
    print(f"  {t:<14} {n:3d}")

Conflicts: 201    |    Panel: 1989-01 – 2024-12

Agreement timing breakdown:
  never_signed    97
  during_ged      82
  pre_ged         11
  post_ged        11


### **Pre-GED agreements — reclassify to non-signer**
Agreements dated **before `start_date`** fall outside the observable GED spell. These conflicts had a formal agreement, but the GED-recorded fighting started *after* that agreement. Within our survival model, they never entered the risk set as potential signers. They are reclassified as **non-signers** (`ever_agreement = 0`).

In [182]:
# ── Pre-GED: display and fix ─────────────────────────────────────────────────
pre_ged = cl[cl['agree_timing'] == 'pre_ged'].copy()
pre_ged['months_before_onset'] = (
    pre_ged['start_date_numeric'] - pre_ged['first_agreement_date']
).astype(int)

print(f"Pre-GED conflicts: {len(pre_ged)}")
print()
display(
    pre_ged[['conflict_id', 'country', 'start_date', 'end_date',
             'pax_first_date', 'months_before_onset']]
    .sort_values('months_before_onset', ascending=False)
    .reset_index(drop=True)
)

# Fix: reclassify to non-signer
cl.loc[cl['agree_timing'] == 'pre_ged', 'ever_agreement'] = 0
print(f"ever_agreement after pre-GED fix: "
      f"0 → {(cl['ever_agreement']==0).sum()},  "
      f"1 → {(cl['ever_agreement']==1).sum()}")

Pre-GED conflicts: 11



,conflict_id,country,start_date,end_date,pax_first_date,months_before_onset
0,274,India,2020-06,2020-08,1993-09-01,321
1,11345,South Sudan,2011-08,2024-08,2006-01-01,67
2,416,Central African Republic,2001-05,2024-12,1997-01-01,52
3,402,Yemen (North Yemen),1994-02,1994-07,1990-04-01,46
4,329,Ethiopia,1993-10,2018-01,1991-07-01,27
5,230,Yemen (North Yemen),2009-11,2024-12,2008-02-01,21
6,390,Bosnia-Herzegovina,1992-04,1995-09,1991-09-01,7
7,11344,Sudan,2011-05,2011-06,2011-01-01,4
8,388,Azerbaijan,1991-12,2023-12,1991-09-01,3
9,13306,Ukraine,2014-08,2022-02,2014-06-01,2


ever_agreement after pre-GED fix: 0 → 108,  1 → 93


---
## **2. Termination Dataset: extract last episode per conflict**

The **UCDP Conflict Termination Dataset** records the outcome of each conflict episode.
Row structure:
- Each **row = one calendar year** within one episode
- An **episode** = a continuous period with ≥ 25 annual battle deaths
- `c_epterm = 1` only on the **terminal year** of a terminated episode; `c_ependdate`
  and `c_outcome` are populated only on those rows
- All other rows have `c_epterm = 0` (or NaN) and `c_ependdate = NaN`

We take the **row with the maximum year** per `conflict_id`:
- If that row has `c_epterm = 0 / NaN` → last episode is ongoing → `c_ependdate = NaN`
- If that row has `c_epterm = 1` → last episode terminated → `c_ependdate` and `c_outcome` available

In [183]:
# ── Extract last row by max year per conflict_id ──────────────────────────────
df_term = pd.read_csv(TERM_PATH, low_memory=False)

OUTCOME_LABEL_MAP = {
    1.0: 'Peace agreement',  2.0: 'Ceasefire',
    3.0: 'Victory (govt)',   4.0: 'Victory (rebels)',
    5.0: 'Low activity',     6.0: 'Actor ceases to exist'
}

last_term = (
    df_term[df_term['conflict_id'].isin(cl['conflict_id'].unique())]
    .sort_values(['conflict_id', 'year'])
    .drop_duplicates('conflict_id', keep='last')
    [['conflict_id', 'year', 'c_ependdate', 'c_outcome', 'c_epterm']]
    .rename(columns={
        'year':        'year_term_out',
        'c_ependdate': 'c_ependdate_term_out',
        'c_outcome':   'c_outcome_term_out',
        'c_epterm':    'c_epterm_term_out',
    })
)
last_term['c_ependdate_term_out'] = last_term['c_ependdate_term_out'].replace(' ', np.nan)
n_ongoing = last_term['c_ependdate_term_out'].isna().sum()
n_term    = last_term['c_ependdate_term_out'].notna().sum()

print(f"Conflicts covered: {len(last_term)}")
print(f"  Last episode ongoing   (c_ependdate = NaN):   {n_ongoing}")
print(f"  Last episode terminated (c_ependdate filled): {n_term}")
print()
print("Termination outcome — terminated last episodes:")
print(
    last_term[last_term['c_ependdate_term_out'].notna()]
    ['c_outcome_term_out'].map(OUTCOME_LABEL_MAP).value_counts().to_string()
)

Conflicts covered: 201
  Last episode ongoing   (c_ependdate = NaN):   73
  Last episode terminated (c_ependdate filled): 128

Termination outcome — terminated last episodes:
c_outcome_term_out
Low activity             47
Victory (govt)           27
Peace agreement          18
Ceasefire                16
Victory (rebels)         10
Actor ceases to exist    10


---
## **3. Merge and assign termination outcome**

We merge and compare:
- `end_year_ged` = year of `end_date` (last GED event)
- `end_year_term` = year of `c_ependdate_term_out` (last termination date)

**Decision rule for `termination_outcome_label`:**

| Condition | Label |
|---|---|
| `c_ependdate_term_out` is NaN | `"ongoing"` |
| `end_year_ged == end_year_term` and `c_outcome` not NaN | c_outcome label (e.g. `"Victory (govt)"`) |
| `end_year_ged == end_year_term` and `c_outcome` NaN | `"ongoing"` |
| `end_year_ged > end_year_term` (GED extends beyond Termination) | `"ongoing"` |

**Post-GED agreement rule** (applied after assigning `termination_outcome_label`):

| Condition | Action |
|---|---|
| `agree_timing == 'post_ged'` AND `termination_outcome_label == 'ongoing'` | Keep `ever_agreement = 1` (agreement is the exit) |
| `agree_timing == 'post_ged'` AND `termination_outcome_label != 'ongoing'` | Set `ever_agreement = 0` (competing event dominates) |

In [184]:
# ── Merge panel with termination data ────────────────────────────────────────

cl = cl.merge(last_term, on='conflict_id', how='left')

cl['end_year_ged']  = cl['end_date'].str[:4].astype(int)
cl['end_year_term'] = pd.to_datetime(
    cl['c_ependdate_term_out'], errors='coerce'
).dt.year

def assign_label(row):
    end_term = row['end_year_term']
    end_ged  = row['end_year_ged']
    outcome  = row['c_outcome_term_out']
    if pd.isna(end_term):
        return 'ongoing'
    if end_ged == end_term:
        return OUTCOME_LABEL_MAP.get(outcome, 'ongoing') if not pd.isna(outcome) else 'ongoing'
    return 'ongoing'  # GED extends beyond Termination endpoint

cl['termination_outcome_label'] = cl.apply(assign_label, axis=1)

print("termination_outcome_label distribution:")
print(cl['termination_outcome_label'].value_counts().sort_index().to_string())

termination_outcome_label distribution:
termination_outcome_label
Actor ceases to exist      5
Ceasefire                  5
Low activity              13
Peace agreement            9
Victory (govt)            19
Victory (rebels)           9
ongoing                  141


In [185]:
# ── Year comparison: match vs. no-match ──────────────────────────────────────

has_ep   = cl['end_year_term'].notna()
match    = has_ep & (cl['end_year_ged'] == cl['end_year_term'])
no_match = has_ep & (cl['end_year_ged'] != cl['end_year_term'])

print(f"No termination endpoint (ongoing episode):    {(~has_ep).sum():3d}  → 'ongoing'")
print(f"With endpoint — years MATCH:                  {match.sum():3d}  → c_outcome label")
print(f"With endpoint — years NO MATCH (GED > Term):  {no_match.sum():3d}  → 'ongoing'")
print()
print("Matching conflicts — termination outcome breakdown:")
print(cl[match]['c_outcome_term_out'].map(OUTCOME_LABEL_MAP).value_counts().to_string())
print()
nm = cl[no_match].copy()
nm['year_gap'] = nm['end_year_ged'] - nm['end_year_term']
print("Non-matching — c_outcome and year gap summary:")
print(nm['c_outcome_term_out'].map(OUTCOME_LABEL_MAP).value_counts().to_string())
print(f"  year gap: mean={nm['year_gap'].mean():.1f},  "
      f"median={nm['year_gap'].median():.0f},  max={nm['year_gap'].max():.0f}")

No termination endpoint (ongoing episode):     73  → 'ongoing'
With endpoint — years MATCH:                   60  → c_outcome label
With endpoint — years NO MATCH (GED > Term):   68  → 'ongoing'

Matching conflicts — termination outcome breakdown:
c_outcome_term_out
Victory (govt)           19
Low activity             13
Victory (rebels)          9
Peace agreement           9
Ceasefire                 5
Actor ceases to exist     5

Non-matching — c_outcome and year gap summary:
c_outcome_term_out
Low activity             34
Ceasefire                11
Peace agreement           9
Victory (govt)            8
Actor ceases to exist     5
Victory (rebels)          1
  year gap: mean=6.9,  median=4,  max=35


In [186]:
print("Top 15 non-matching conflicts by year gap --> reclassified to 'ongoing' in survival model")
print(
    nm[['conflict_id', 'country', 'end_year_ged', 'end_year_term', 'year_gap']]
    .assign(c_outcome=nm['c_outcome_term_out'].map(OUTCOME_LABEL_MAP))
    .sort_values('year_gap', ascending=False)
    .head(15)
    .to_string(index=False)
)

Top 15 non-matching conflicts by year gap --> reclassified to 'ongoing' in survival model
 conflict_id               country  end_year_ged  end_year_term  year_gap             c_outcome
         331               Morocco          2024         1989.0      35.0          Low activity
         322            Bangladesh          2022         1991.0      31.0             Ceasefire
         224       Myanmar (Burma)          2019         1996.0      23.0        Victory (govt)
         251                 India          2023         2000.0      23.0             Ceasefire
         315        United Kingdom          2019         1998.0      21.0             Ceasefire
         342                 Spain          2009         1991.0      18.0          Low activity
         422       Myanmar (Burma)          2015         1997.0      18.0      Victory (rebels)
         392               Georgia          2011         1993.0      18.0       Peace agreement
         335                 India          20

In [187]:
# ── Crosstab before post-GED fix ─────────────────────────────────────────────

print("Crosstab — termination_outcome_label × ever_agreement (before post-GED fix):")
ct = pd.crosstab(
    cl['termination_outcome_label'],
    cl['ever_agreement'],
    margins=True
).rename(columns={0: 'Never signed', 1: 'Signed PA-X', 'All': 'Total'})
display(ct)

Crosstab — termination_outcome_label × ever_agreement (before post-GED fix):


ever_agreement,Never signed,Signed PA-X,Total
termination_outcome_label,,,
Actor ceases to exist,4,1,5
Ceasefire,1,4,5
Low activity,7,6,13
Peace agreement,1,8,9
Victory (govt),14,5,19
Victory (rebels),6,3,9
ongoing,75,66,141
All,108,93,201


### Post-GED agreements — apply termination-based rule

For the conflicts where `agree_timing == 'post_ged'`, the agreement arrived after the last GED event. We now know their `termination_outcome_label`:
- **Ongoing** → the conflict was still active when the agreement was reached → `ever_agreement = 1`
- **Not ongoing** (victory, fade, etc.) → the conflict had already exited the risk set → `ever_agreement = 0`

In [188]:
# ── Post-GED fix ─────────────────────────────────────────────────────────────

post_ged       = cl['agree_timing'] == 'post_ged'
post_ongoing   = post_ged & (cl['termination_outcome_label'] == 'ongoing')
post_competing = post_ged & (cl['termination_outcome_label'] != 'ongoing')

print(f"Post-GED conflicts: {post_ged.sum()}")
print(f"  termination = 'ongoing'  → keep ever_agreement = 1:  {post_ongoing.sum()}")
print(f"  termination ≠ 'ongoing'  → set  ever_agreement = 0:  {post_competing.sum()}")

if post_competing.sum() > 0:
    print()
    print("Post-GED conflicts reclassified → ever_agreement = 0:")
    display(
        cl[post_competing][
            ['conflict_id', 'country', 'end_date', 'pax_first_date',
             'termination_outcome_label']
        ].reset_index(drop=True)
    )
    cl.loc[post_competing, 'ever_agreement'] = 0

if post_ongoing.sum() > 0:
    print()
    print("Post-GED conflicts kept as signers (termination = 'ongoing'):")
    display(
        cl[post_ongoing][
            ['conflict_id', 'country', 'end_date', 'pax_first_date',
             'termination_outcome_label']
        ].reset_index(drop=True)
    )

Post-GED conflicts: 11
  termination = 'ongoing'  → keep ever_agreement = 1:  2
  termination ≠ 'ongoing'  → set  ever_agreement = 0:  9

Post-GED conflicts reclassified → ever_agreement = 0:


,conflict_id,country,end_date,pax_first_date,termination_outcome_label
0,260,Lebanon,1990-10,1991-05-01,Victory (govt)
1,271,Iraq,1996-09,2009-06-01,Low activity
2,298,South Africa,1989-05,1991-05-01,Peace agreement
3,307,Guinea,2001-07,2010-01-01,Low activity
4,371,Iraq,1991-03,1991-04-01,Victory (rebels)
5,407,Comoros,1997-09,1997-12-01,Victory (rebels)
6,411,Lesotho,1998-09,1999-12-01,Victory (govt)
7,420,Iraq,2003-04,2003-11-01,Victory (govt)
8,435,Djibouti,2008-06,2010-06-01,Low activity



Post-GED conflicts kept as signers (termination = 'ongoing'):


,conflict_id,country,end_date,pax_first_date,termination_outcome_label
0,425,Nigeria,2009-04,2016-07-01,ongoing
1,11475,Myanmar (Burma),2011-09,2012-04-01,ongoing


In [189]:
print("=== Final crosstab — termination_outcome_label × ever_agreement ===")
ct_final = pd.crosstab(
    cl['termination_outcome_label'],
    cl['ever_agreement'],
    margins=True
).rename(columns={0: 'Never signed', 1: 'Signed PA-X', 'All': 'Total'})
display(ct_final)

causes_map = {
    'ongoing':               'censored',
    'Peace agreement':       'fade',
    'Ceasefire':             'fade',
    'Low activity':          'fade',
    'Victory (govt)':        'not at risk',
    'Victory (rebels)':      'not at risk',
    'Actor ceases to exist': 'not at risk',
}

cl['cause_label'] = np.where(
    cl['ever_agreement'] == 1,
    'first_agreement',
    cl['termination_outcome_label'].map(causes_map)
)

print(f"Signers       (ever_agreement = 1): {(cl['ever_agreement']==1).sum()}")
print(f"Non-signers   (ever_agreement = 0): {(cl['ever_agreement']==0).sum()}")
print()
print(cl['cause_label'].value_counts())

=== Final crosstab — termination_outcome_label × ever_agreement ===


ever_agreement,Never signed,Signed PA-X,Total
termination_outcome_label,,,
Actor ceases to exist,4,1,5
Ceasefire,1,4,5
Low activity,10,3,13
Peace agreement,2,7,9
Victory (govt),17,2,19
Victory (rebels),8,1,9
ongoing,75,66,141
All,117,84,201


Signers       (ever_agreement = 1): 84
Non-signers   (ever_agreement = 0): 117

cause_label
first_agreement    84
censored           75
not at risk        29
fade               13
Name: count, dtype: int64


In [190]:
# # ── Export conflict-level outcome table ──────────────────────────────────────

# export_cols = [
#     'conflict_id', 'country', 'isocode',
#     'start_date', 'end_date', 'pax_first_date',
#     'ever_agreement', 'agree_timing',
#     'termination_outcome_label',
#     'end_year_ged', 'end_year_term',
#     'c_outcome_term_out', 'c_ependdate_term_out',
# ]

# out = cl[export_cols].copy()
# out.to_csv(OUT_PATH, index=False)
# print(f"Exported → {OUT_PATH}  ({len(out)} rows)")
# print()
# print(out['termination_outcome_label'].value_counts().to_string())